# Run Compute Tasks

Instructions:

1. Replace the contents of the variables in the following cell
2. If your function requires arguments, set `f_args` and `f_kwargs` in the cell after next
3. Run the final cell and await your results

In [ ]:
globus_compute_tutorial_endpoint_id = "4b116d3c-1703-4f8f-9f6f-39921e5864df"
your_endpoint_id = "<put_your_ep_uuid_here>"

endpoint_id = globus_compute_tutorial_endpoint_id  # Update which endpoint_id to use

# Replace this string with the function to call within the module ...
module_entrypoint = "pretty_print_peek_at_host"

module_desc = "[optional] describe the function's intention or other useful-to-you-later details"

# ... and define that module within this string.  (i.e. replace the content with your code)
python_module_text = """
import os
import typing as t

import psutil

def row_by_row(data: t.Iterable[tuple[t.Any, t.Any]]) -> str:
    lines = []
    for ndx, (key, val) in enumerate(data):
        c = ndx % 2 and "\033[92;1m" or "\033[93;1m"
        lines.append(f"{c}{key:>19}: {val}\033[0m")
    return "\\n".join(lines)


def peek_at_host() -> dict[str, t.Any]:
    return {
      "core_count": os.cpu_count(),
      "physical_core_count": psutil.cpu_count(logical=False),
      "cores_available": os.sched_getaffinity(0),
      "virtmem": psutil.virtual_memory(),
      "uname": os.uname(),
    }


def pretty_print_peek_at_host() -> str:
    return row_by_row(peek_at_host().items())
"""
assert f"def {module_entrypoint}" in python_module_text, "Entrypoint not found!"

In [ ]:
# If the function needs arguments, specify them here
f_args = ()
f_kwargs = {}

In [ ]:
import time
from globus_compute_sdk import Executor

with Executor(endpoint_id) as ex:
    func_id = ex.register_source_code(python_module_text, module_entrypoint, description=module_desc)
    future = ex.submit_to_registered_function(func_id, args=f_args, kwargs=f_kwargs)
    print("Task enqueued, awaiting submission.")
    try:
        while not future.task_id:
            time.sleep(0.1)
        print(f"Task submitted ({future.task_id}); awaiting result:")
        print(future.result())
    except Exception as e:
        print("  \033[91;1mOh no!  Task failed.  The reported exception was\033[0m:")
        print(str(e))